# 1장 — 큰 그림: 검색의 진화와 RAG 🗺️

웹 챕터의 두 실험을 코드로 다시 만납니다 — 이번엔 **미니 RAG 파이프라인을 끝까지 조립**해 봅니다.

**이 노트북의 목표**
1. 같은 문서 5건 위에서 **키워드 검색 vs 의미 검색**의 승자가 질문마다 다름을 확인
2. **RAG 5단계**(질문 → 임베딩 → 검색 → 프롬프트 조립 → 생성)를 함수 하나씩으로 조립

> 임베딩은 미리 준비한 2차원 toy 좌표를 씁니다(오프라인·결정적). 진짜 임베딩과 진짜 LLM 은 10장 캡스톤에서.

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch1')

## 1. 검색 대상 — 사내 문서 5건과 두 개의 질문

LLM 훈련 데이터에는 없는, 우리 회사만 아는 문서들입니다(지식 마감선 밖).

In [ ]:
import math

# 사내 문서 5건 — (제목, 본문, 미리 준비한 2D toy 임베딩 좌표)
DOCS = {
    'A': ('연차 신청 안내',     '인사 포털에서 결재선 지정 후 제출',      (2.0, 8.0)),
    'B': ('VPN 오류 코드 403',  '사내망 인증서 갱신 필요',               (8.0, 7.5)),
    'C': ('법인카드 정산 안내', '매월 5일까지 경비 시스템에 등록',        (3.0, 2.0)),
    'D': ('회의실 예약 방법',   '그룹웨어 캘린더에서 예약',              (6.0, 3.0)),
    'E': ('프린터 오류 해결',   '용지 걸림 확인 후 재시도',              (7.2, 6.3)),
}

# 질문 2개 — 좌표는 뜻 기준으로 미리 배치 (좌표를 '누가' 정하는지는 9장)
Q1 = ('쉬는 날 미리 쓰려면 어디에 얘기해야 하나요?', (2.3, 7.6))   # 표현이 완전히 다른 질문 → 정답 A
Q2 = ('VPN 403 이라고 떠요',                       (7.8, 7.2))   # 정확한 용어·코드 → 정답 B

for k, (title, body, _) in DOCS.items():
    print(f'  {k}. {title} — {body}')

## 2. 실험 ① — 같은 질문, 두 검색기

1세대 **키워드 검색**(겹치는 단어 수)과 2세대 **의미 검색**(지도에서 가까움)을 나란히 돌립니다.

In [ ]:
def keyword_search(question):
    """1세대: 질문과 겹치는 단어 수로 순위 매기기 (실무 표준판의 이름은 BM25)."""
    q_words = set(question.split())
    scores = {k: len(q_words & set((t + ' ' + b).split())) for k, (t, b, _) in DOCS.items()}
    return sorted(scores, key=scores.get, reverse=True), scores

def semantic_search(q_pos):
    """2세대: 의미 지도에서 질문과 가까운 순 (자의 업그레이드는 2장 — 코사인)."""
    dist = {k: math.dist(q_pos, pos) for k, (_, _, pos) in DOCS.items()}
    return sorted(dist, key=dist.get), dist

for name, (q_text, q_pos) in [('Q1 (표현이 다른 질문)', Q1), ('Q2 (정확한 코드)', Q2)]:
    kw_rank, kw = keyword_search(q_text)
    sem_rank, _ = semantic_search(q_pos)
    print(f'{name}: "{q_text}"')
    print(f'   🔤 키워드 1등: {kw_rank[0]} (겹침 {kw[kw_rank[0]]}개)   🗺️ 의미 1등: {sem_rank[0]}\n')

# 강점 지형의 확인 — 승자가 질문마다 다르다!
kw1_rank, kw1 = keyword_search(Q1[0])
assert max(kw1.values()) == 0, 'Q1 은 겹치는 단어가 0 — 키워드 검색은 정답이 있어도 못 찾는다'
assert semantic_search(Q1[1])[0][0] == 'A', 'Q1 의 의미 검색 1등은 A(연차 안내)여야'
kw2_rank, kw2 = keyword_search(Q2[0])
assert kw2_rank[0] == 'B' and kw2['B'] >= 2, 'Q2 는 정확한 용어가 꽂혀 키워드가 B 를 확실히 찾는다'
print('✔ 확인: Q1 은 의미 검색만 정답, Q2 는 키워드가 확실 — 그래서 실무는 하이브리드(병용)')

## 3. 실험 ② — 미니 RAG 파이프라인 조립

5단계를 함수 하나씩으로 만들고 이어 붙입니다. **모델 재훈련은 어디에도 없다**는 걸 눈여겨보세요.

In [ ]:
def embed(question):
    """2단계 — 질문을 좌표로. (여기선 준비된 좌표를 찾아 쓰고, 실제는 모델이 계산 — 9장)"""
    return {Q1[0]: Q1[1], Q2[0]: Q2[1], '법인카드 언제까지 정산해야 하나요?': (3.2, 2.3)}[question]

def retrieve(q_pos, top_k=1):
    """3단계 — 지도에서 가까운 문서 top-k 를 채택. (몇 개 넣을지는 7장, 빨리 찾기는 6장)"""
    rank, _ = semantic_search(q_pos)
    return rank[:top_k]

def build_prompt(question, doc_keys):
    """4단계 — 찾은 문서를 질문 앞에 붙인다. 지식이 문맥 창에 실리는 순간(6부 2장)."""
    docs_text = '\n'.join(f'[문서] {DOCS[k][0]} — {DOCS[k][1]}' for k in doc_keys)
    return f'아래 사내 문서를 참고해 답하세요.\n{docs_text}\n[질문] {question}'

def generate(prompt):
    """5단계 — LLM 의 답 생성. (여기선 흉내 — 진짜 LLM 연결은 10장 캡스톤)"""
    doc_line = [l for l in prompt.split('\n') if l.startswith('[문서]')][0]
    title, body = doc_line.removeprefix('[문서] ').split(' — ')
    return f'{body}하시면 됩니다. (출처: {title})'

# ── 조립: 질문 하나가 답이 되기까지 ──
question = '법인카드 언제까지 정산해야 하나요?'
q_pos = embed(question)                 # ② 임베딩
picked = retrieve(q_pos, top_k=1)       # ③ 검색
prompt = build_prompt(question, picked) # ④ 프롬프트 조립
answer = generate(prompt)               # ⑤ 생성

print(f'① 질문   : {question}')
print(f'② 임베딩 : {q_pos}')
print(f'③ 검색   : {picked} ({DOCS[picked[0]][0]})')
print(f'④ 프롬프트:\n{prompt}\n')
print(f'⑤ 답변   : {answer}')

assert picked == ['C'], '정산 질문의 최근접 문서는 C(법인카드 정산 안내)여야'
assert '5일' in answer and '출처' in answer, '답에 회사 지식(5일)과 출처가 실려야 — RAG 의 두 강점'
print('\n✔ 확인: 재훈련 없이 회사 지식이 답에 들어왔다 — 문서를 고치면 다음 답부터 즉시 반영')

**직접 해보기**: 위 셀의 `DOCS['C']` 본문을 `'매월 10일까지 …'` 로 바꾸고 다시 실행해 보세요.
재훈련 없이 답이 바로 바뀝니다 — 조련(파인튜닝)과의 결정적 차이.

## 4. 관찰 정리 + 🤖 AI 연결

- **강점 지형**: 표현이 다른 질문 = 의미 검색, 정확한 용어·코드 = 키워드(BM25). 실무는 하이브리드.
- **RAG** (검색 증강 생성, Retrieval-Augmented Generation) = 질문 → 임베딩 → 검색 → 프롬프트 조립 → 생성.
  지식은 가중치가 아니라 **프롬프트(문맥 창)** 에 실린다 — 그래서 갱신이 즉시고 출처를 달 수 있다.
- **두 길의 분업**: 말투·행동 양식은 조련(6부, 파인튜닝), 사실·최신 지식은 RAG(7부). 선택 기준 심화는 8장.
- 파이프라인의 단계마다 수학 질문이 하나씩: 가까움의 자(2장 코사인), 공간의 성질(3~5장),
  빨리 찾기(6장 벡터 DB), 무엇을 넣나(7장 청킹·top-k), 조립·실패 모드(8장), 좌표 학습(9장).

**다음 챕터**: 2장 — 가까움의 수학, 코사인 유사도 (첫 핵심). 오늘 쓴 `math.dist` 에는 함정이 있습니다 —
**문서 길이가 점수를 오염**시켜요. 1부 2장의 내적이 그 해결사로 되살아납니다.